# Programming for Deep Reinforcement Learning

Lab Course: Challenging Problems in Reinforcement Learning

In this notebook, we focus on two things that can be useful to your project work and that you will encounter a lot when programming for (Deep) Reinforcement Learning (RL) -- and also general Machine Learning.
1. Implementing a Deep RL algorithm in ```Pytorch```
2. Using Deep RL algorithms from the ```stable_baselines3``` library

Additionally, for those interested in safety in RL, we include
3. Safe RL training using ```omnisafe``` and ```safety_gymnasium```

Please fill out the code everywhere it says `# TODO: ...`.

## Installing the relevant libraries

Run the cell below if you don't have all necessary libraries installed.

In [ ]:
!uv pip install torch gymnasium stable-baselines3 imageio

## Deep Q-Learning in Pytorch

In the first notebook, we used (standard, tabular) Q-learning on two gymnasium environments. When we combine Q-learning with neural networks to represent the policy, the resulting algorithm is called DQN (Deep Q-learning). DQN is one of the most fundamental and simple algorithms in DRL. Please check the original paper [here](https://arxiv.org/pdf/1312.5602) for details on how it works!

We can implement DQN (and in general, neural network based Machine Learning + RL algorithms) easily in ```pytorch```. ```pytorch``` is one of the major frameworks used for this task (there exist more e.g. Tensorflow, JAX), so understanding how to use it is a good skill to have if you like ML!

In this part of the notebook, we use ```pytorch``` to implement DQN. Disclaimer: DQN is a fundamental, well-known algorithm. There are many implementations available online. Therefore, we largely use code from [this](https://pytorch.org/tutorials/intermediate/reinforcement_q_learning.html) pytorch tutorial here. The purpose is simply to introduce you to ML/RL coding in ```pytorch```.

Additionally, we include the pseudo-code of DQN, taken from the original paper (linked above), here. Take a look and compare it to the tabular Q-learning you used last week! The major differences here lie in the use of a replay memory to store observations in the form of tuples (state, action, next state, reward), that the model is trained on, and in the use of (two) neural networks to represent the policy, instead of a Q-table.

[<img src="DQN_algorithm.png" width="750"/>](DQN_algorithm.png)

In [ ]:
# First, we import the necessary libraries
import gymnasium as gym
import math
import random
import matplotlib.pyplot as plt
from IPython import display
from collections import namedtuple, deque
import imageio
from IPython.display import Image
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Then, like last week, we create our gym environment.
# TODO: create an environment for the "CartPole-v1" task!
""" START COCE """
env = gym.make("CartPole-v1", render_mode="rgb_array")
""" END CODE """

In [ ]:
# The replay memory (or replay buffer) stores transitions in the form of (state, action, next_state, reward)
# In every step, the current transition is added to the replay memory.
# During learning, batches of transition are sampled from the memory and used to update the policy
# The replay memory is not specific to DQN, but used in many DRL algorithms.
# We provide you with the data structure here.

Transition = namedtuple('Transition',
                        ('state', 'action', 'next_state', 'reward'))


class ReplayMemory(object):

    def __init__(self, capacity):
        self.memory = deque([], maxlen=capacity)

    def push(self, state, action, next_state, reward):
        """Save a transition"""
        self.memory.append(Transition(state, action, next_state, reward))

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)

In [ ]:
# This class is used for the neural network representation of the Q function.
# It is based on the pytorch class nn.Module - this class is the basis of all Pytorch neural network classes.
# It needs at least two main functions: __init__ is used to initialise the layers of the network,
# and forward "forwards" the input through the layers of the network:
# It takes the state of the RL environment as input, and predicts the next action the agent should take.

# Disclaimer: We renamed this class from the Pytorch Tutorial, from DQN to QNetwork, since it does not represent the entire algorithm, only the Q function!

class QNetwork(nn.Module):

    def __init__(self, n_observations, n_actions, n_hidden):
        super(QNetwork, self).__init__()
        self.layer1 = nn.Linear(n_observations, n_hidden)
        self.layer2 = nn.Linear(n_hidden, n_hidden)
        # TODO: add the third (output) layer using the right dimensions!
        """ START CODE """
        self.layer3 = nn.Linear(n_hidden, n_actions)
        """ END CODE """

    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        return self.layer3(x)

In [ ]:
# We use this function to plot the episode reward during and after training.
episode_rewards = []

def plot_rewards(show_result=False):
    plt.figure(1)
    if show_result:
        plt.title('Result')
    else:
        plt.clf()
        plt.title('Training...')
    plt.xlabel('Episode')
    plt.ylabel('Reward')
    plt.plot(episode_rewards)

    plt.pause(0.001)
    if not show_result:
        display.display(plt.gcf())
        display.clear_output(wait=True)
    else:
        display.display(plt.gcf())

In [ ]:
# Now we will set some hyperparameters for training
# Feel free to change values and watch the observe the effect they have on training!

BATCH_SIZE = 128        # the number of transitions sampled from the replay buffer
GAMMA = 0.99            # the discount factor
EPS_START = 0.9         # the value of epsilon at the beginning of training
EPS_END = 0.05          # the final value of epsilon at the end of training
EPS_DECAY = 1000        # controls the rate of exponential decay of epsilon, higher means a slower decay
TAU = 0.005             # update rate of the target network
LR = 1e-4               # the learning rate
N_HIDDEN = 128          # number of neurons in the hidden layer -> check out the QNetwork class!
MEMORY_CAPACITY = 10000 # size of the replay memory

# Get number of actions from gym action space
n_actions = env.action_space.n
# Get the number of state observations
state, info = env.reset()
n_observations = len(state)

NUM_EPISODES = 500

In [ ]:
# Now, we implement the DQN algorithm itself, following the pseudo-code from the original paper.

## Algorithm 1 Deep Q-learning with Experience Replay

## Initialize replay memory D to capacity N
memory = ReplayMemory(MEMORY_CAPACITY)
## Initialize action-value function Q with random weights
q_network = QNetwork(n_observations, n_actions, N_HIDDEN)
# TODO: initialize a target network with the same dimensions as the q_network!
""" START CODE """
target_net = QNetwork(n_observations, n_actions, N_HIDDEN)
""" END CODE """
target_net.load_state_dict(q_network.state_dict())

optimizer = torch.optim.AdamW(q_network.parameters(), lr=LR, amsgrad=True)

steps_done = 0

## for episode 1, M do
for i_episode in range(NUM_EPISODES):
    episode_reward = 0
##   Initialise sequence s_1 = {x_1} and preprocessed sequenced phi_1 = phi(s_1)
    # TODO: get the initial state from the environment using the reset() function
    """ START CODE """
    state, info = env.reset()
    """ END CODE """
    state = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
##   for t = 1, T do
    while True:
##       With probability epsilon select a random action a_t
##       otherwise select a_t = max_a Q*(phi(s_t), a; theta)
        sample = random.random()
        eps_threshold = EPS_END + (EPS_START - EPS_END) * math.exp(-1. * steps_done / EPS_DECAY)
        steps_done += 1
        # TODO: implement epsilon-greedy - remember from the previous notebook, or look at the pseudo-code!
        # For the random action, use this: torch.tensor([[env.action_space.sample()]], dtype=torch.long)
        # For the action based on the policy, use this: q_network(state).max(1).indices.view(1,1)
        """ START CODE """
        if sample <= eps_threshold:
            with torch.no_grad():
                action = torch.tensor([[env.action_space.sample()]], dtype=torch.long)
        else:
            action = q_network(state).max(1).indices.view(1,1)
        """ END CODE """
##       Execute action a_t in emulator and observe reward r_t and image x_{t+1}
##       Set s_{t+1} = s_t, a_t, x_{t+1} and preprocess phi_{t+1} = phi(s_{t+1})

        # TODO: using the action.item(), take a step in the environment and observe the observation, reward, terminated, truncated, _.
        observation, reward, terminated, truncated, _ = env.step(action.item())
        episode_reward += reward
        reward = torch.tensor([reward])
        done = terminated or truncated

        if terminated:
            next_state = None
        else:
            next_state = torch.tensor(observation, dtype=torch.float32).unsqueeze(0)

##       Store transition (phi_t, a_t, r_t, phi_{t+1}) in D
        # TODO: add the relevant information to the replay buffer!
        """ START CODE """
        memory.push(state, action, next_state, reward)
        """ END CODE """
        state = next_state

        if len(memory) >= BATCH_SIZE:
##       Sample random minibatch of transitions (phi_j, a_j, r_j, phi_{j+1}) from D
            # TODO: sample a batch of transitions of size BATCH_SIZE from the memory
            """ START CODE """
            transitions = memory.sample(BATCH_SIZE)
            """ END CODE """
            batch = Transition(*zip(*transitions))
##       Set y_j =   {r_j, for terminal phi_{j+1}
##                   {r_j + gamma * max_a' Q(phi_{j+1}, a'; theta), for non-terminal phi_{j+1}
##       Perform a gradient descent step on (y_j - Q(phi_j, a_j; theta))**2 according to equation 3
            #optimize_model()

            # Compute a mask of non-final states and concatenate the batch elements
            non_final_mask = torch.tensor(tuple(map(lambda s: s is not None,
                                                batch.next_state)), dtype=torch.bool)
            non_final_next_states = torch.cat([s for s in batch.next_state
                                                        if s is not None])
            state_batch = torch.cat(batch.state)
            action_batch = torch.cat(batch.action)
            reward_batch = torch.cat(batch.reward)

            # Compute Q(s_t, a) - the model computes Q(s_t), then we select the columns of actions taken
            # That means that for each state in the batch, we receive the action that we would have taken according to the policy
            state_action_values = q_network(state_batch).gather(1, action_batch)

            # Here, we compute the next state values accordin gto the target network
            next_state_values = torch.zeros(BATCH_SIZE)
            with torch.no_grad():
                next_state_values[non_final_mask] = target_net(non_final_next_states).max(1).values
            expected_state_action_values = (next_state_values * GAMMA) + reward_batch

            # Compute Huber loss
            criterion = nn.SmoothL1Loss()
            loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))

            # Optimize the model
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_value_(q_network.parameters(), 100)
            optimizer.step()

            # Update the target network
            target_net_state_dict = target_net.state_dict()
            q_net_state_dict = q_network.state_dict()
            for key in q_net_state_dict:
                target_net_state_dict[key] = q_net_state_dict[key] * TAU + target_net_state_dict[key] * (1-TAU)
            target_net.load_state_dict(target_net_state_dict)

        if done:
            plot_rewards()
            episode_rewards.append(episode_reward)
            break

plot_rewards(show_result=True)
plt.ioff()
plt.show()

In [ ]:
def record_video(env, QNetwork, out_directory, fps=1):
  """
  Generate a replay video of the agent
  :param env
  :param Qnetwork: the Qnetwork of our agent
  :param out_directory
  :param fps: how many frame per seconds
  """
  images = []
  observation, info = env.reset()
  img = env.render()
  images.append(img)
  while True:
    # Take the action (index) that have the maximum expected future reward given that state
    observation_tensor = torch.tensor(observation, dtype=torch.float32).unsqueeze(0)
    action = QNetwork(observation_tensor).max(1).indices.view(1,1)
    observation, reward, terminated, truncated, _ = env.step(action.item())

    img = env.render()
    images.append(img)

    if terminated or truncated:
        break
  imageio.mimsave(out_directory, [np.array(img) for i, img in enumerate(images)], fps=fps)

In [ ]:
record_video(env, q_network, '/tmp/result_frozen.gif', 10)
Image(open('/tmp/result_frozen.gif','rb').read())

## DRL with stable_baselines3

Luckily, we don't have to implement all standard DRL algorithms ourselves when we want to use them. There are libraries, such as ```stable_baselines3```, that offer many different ready-to-use DRL algorithms, including DQN, which we just implemented.

In the following, we check out at how to train with the ```stable_baselines3``` implementation of DQN, and then use that knowledge to train an other commonly used DRL algorithm, called PPO. Please check out the ```stable_baselines3``` [DQN documentation](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html).

We will not go into the details of PPO here, but please feel free to read up on the ```stable_baselines3``` [PPO documentation](https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html) or take a look at the [original paper](https://arxiv.org/abs/1707.06347).

In [ ]:
# First, we import the libraries: gymnasium for the environment, stable_baselines3 for algorithms
import gymnasium as gym
from stable_baselines3 import DQN, PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.logger import configure, read_csv

In [ ]:
# Creating the environment + agent
env = gym.make("CartPole-v1", render_mode="human")
dqn_model = DQN("MlpPolicy", env, verbose=1)

# To get an easier access to evaluation data, configure the logger to write to a file
dqn_tmp_path = "tmp/sb3_log/dqn"
new_dqn_logger = configure(dqn_tmp_path, ["csv"])
dqn_model.set_logger(new_dqn_logger)

# Training and saving the agent
dqn_model.learn(total_timesteps=50000, log_interval=4)
dqn_model.save("dqn_cartpole")


In [ ]:
# Model loading and evaluation
dqn_model = DQN.load("dqn_cartpole")
dqn_mean_rew, dqn_std_rew = evaluate_policy(dqn_model, env, n_eval_episodes=10)
print(f"DQN evaluation results - mean reward: {dqn_mean_rew}, std. deviation: {dqn_std_rew}")

In [ ]:
# Now, let's look at the training development, by examining the logger data.
# read the logger output
dqn_train_log = read_csv("tmp/sb3_log/dqn/progress.csv")
# take a quick look at the data...
dqn_train_log

In [ ]:
# Let's plot some of the training evaluation!
import matplotlib.pyplot as plt

# First, we plot the reward development in training
plt.title("DQN mean reward during training")
plt.plot(dqn_train_log['time/total_timesteps'], dqn_train_log['rollout/ep_rew_mean'], label="dqn")
plt.legend()
plt.xlabel("# timesteps")
plt.ylabel("mean reward")
plt.show()

# Let's also look at the loss
plt.title("DQN training loss")
plt.plot(dqn_train_log['time/total_timesteps'], dqn_train_log['train/loss'], label="dqn")
plt.legend()
plt.xlabel("# timesteps")
plt.ylabel("training loss")
plt.show()

Now it is your turn! Fill in the code below to train, save, load, and evaluate a PPO model on the cartpole task.

In [ ]:
# TODO
""" START CODE """
# Create the CartPole environment
env = gym.make("CartPole-v1")

# Create a PPO agent
ppo_model = PPO("MlpPolicy", env, verbose=1)

# As with DQN, we save the logger output to a file.
tmp_path_ppo = "tmp/sb3_log/ppo/"
new_ppo_logger = configure(tmp_path_ppo, ["csv"])
ppo_model.set_logger(new_ppo_logger)

# Train the PPO agent for 50000 timesteps (feel free to try more/less!)
ppo_model.learn(total_timesteps=50000, log_interval=4)

# Save the trained model to a file for later use!
ppo_model.save("ppo_cartpole")
""" END CODE """


In [ ]:
# Now, we want to evaluate the trained model that we saved earlier!

# First, let's load the model
ppo_model = PPO.load("ppo_cartpole")

# Use evaluate_policy to receive the mean and standard reward of the trained policy over e.g. 10 episodes.
ppo_mean_rew, ppo_std_rew = evaluate_policy(ppo_model, env, n_eval_episodes=10)

# Print the results
print(f"PPO evaluation results - mean reward: {ppo_mean_rew}, std. deviation: {ppo_std_rew}")

In [ ]:
# TODO: Let's do some visualization again! For that, we consult the log file again.
# read the logger output
""" START CODE """
ppo_train_log = read_csv("tmp/sb3_log/ppo/progress.csv")
""" END CODE """
# take a quick look at the data...
""" START CODE """
ppo_train_log
""" END CODE """


In [ ]:
# TODO: Let's plot the PPO results like before!
# You can re-use the code from above here and add the PPO log data.
# First, we plot the reward development in training
""" START CODE """
plt.title("mean reward during training")
plt.plot(ppo_train_log['time/total_timesteps'], ppo_train_log['rollout/ep_rew_mean'], label="ppo")
plt.legend()
plt.xlabel("# timesteps")
plt.ylabel("mean reward")
plt.show()

# Let's also look at the loss
plt.title("training loss")
plt.plot(ppo_train_log['time/total_timesteps'], ppo_train_log['train/loss'], label="ppo")
plt.legend()
plt.xlabel("# timesteps")
plt.ylabel("training loss")
plt.show()
""" END CODE """

That's it!

Now you saw how easy it is to train DRL models with libraries like ```stable_baselines3```.

## Multi-Objective RL with MO-Gymnasium

In the previous environments we worked with, each step produced a single, scalar reward. In this part of the tutorial, we take the next step and explore environments that provide multiple rewards simultaneously, called Multi-Objective Reinforcement Learning (MO-RL) environments.

[```mo_gymnasium```](https://mo-gymnasium.farama.org/) provides these type of environments. In our task we will use the ```minecart-v0``` environment. In this task the agent learns to steer a minecart, which consumes energy and collects two different types of ore. Check out the documentation on the minecart task [here](https://mo-gymnasium.farama.org/environments/minecart/)!

As we can see the environment has 3 rewards:

1) A positive reward for the first type of ore in range [0, 1.5]
2) A positive reward for the second type of ore also in range [0, 1.5]
3) A negative reward for the energy consumed in range [0, -1]

This makes sense as we want our agent to consume as little energy as possible, so we "punish" it for consuming energy by giving a negative reward.

### Installations and Imports

In [ ]:
!uv pip install stable-baselines3[extra] mo-gymnasium gymnasium pygame

In [ ]:
# --- Imports ---
import mo_gymnasium as mo_gym
from mo_gymnasium.wrappers import LinearReward
import gymnasium as gym

import numpy as np
import matplotlib.pyplot as plt

from stable_baselines3 import DQN, PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.logger import configure, read_csv

### Environment Creation
We can creat the environment the same way as before, but using ```mo_gym``` instead of ```gym```.

However, the RL agents we have been using so far expect a single scalar reward value. To make the environment compatible with these agents, we need to transform the multi-dimensional reward signal. We achieve this by applying the ```LinearReward``` wrapper provided by ```mo_gymnasium```. This wrapper computes the dot product between the reward vector and a weight matrix, producing a single scalar reward that standard RL agents can handle.

In [ ]:
# create the environment
env = mo_gym.make('minecart-v0')

# Scalarize the environment by appliying the weight matrix to the reward vector
env = LinearReward(env, weight=np.array([1, 1, 1]))

### Training the agent

Now you can train an agent for the environment in the same way you did before.

In [ ]:
#TODO: Create and evaluate a PPO agent on the minecart environment

# Create a PPO agent
ppo_model = PPO("MlpPolicy", env, verbose=1)

# Configure Logger.
tmp_path_ppo = "tmp/sb3_log/mo_ppo/"
new_ppo_logger = configure(tmp_path_ppo, ["csv"])
ppo_model.set_logger(new_ppo_logger)

# Train the PPO agent for 50000 timesteps
ppo_model.learn(total_timesteps=50000, log_interval=4)

# Save the trained model to a file for later use!
ppo_model.save("ppo_minecart")

In [ ]:
# Evaluation

# First, let's load the model
ppo_model = PPO.load("ppo_minecart")

# Use evaluate_policy to receive the mean and standard reward of the trained policy over e.g. 10 episodes.
ppo_mean_rew, ppo_std_rew = evaluate_policy(ppo_model, env, n_eval_episodes=10)

# Print the results
print(f"PPO evaluation results - mean reward: {ppo_mean_rew}, std. deviation: {ppo_std_rew}")

In [ ]:
# read the logger output
ppo_train_log = read_csv("tmp/sb3_log/mo_ppo/progress.csv")

# take a quick look at the data...
ppo_train_log

In [ ]:
# First, we plot the reward development in training
plt.title("mean reward during training")
plt.plot(ppo_train_log['time/total_timesteps'], ppo_train_log['rollout/ep_rew_mean'], label="ppo")
plt.legend()
plt.xlabel("# timesteps")
plt.ylabel("mean reward")
plt.show()

# Let's also look at the loss
plt.title("training loss")
plt.plot(ppo_train_log['time/total_timesteps'], ppo_train_log['train/loss'], label="ppo")
plt.legend()
plt.xlabel("# timesteps")
plt.ylabel("training loss")
plt.show()